## Import Libraries


In [ ]:
import os
import gc
from glob import glob
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed

from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## Configuration


In [ ]:
warnings.filterwarnings("ignore")

# Paths
TRAIN_INERTIAL_DIR = "/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/train/inertial_feat/"
TRAIN_VIDEO_DIR = "/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/train/videomae_feat/"
TEST_INERTIAL_PATH = "/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/test/test_inertial_data.npy"
TEST_VIDEO_PATH = "/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/test/test_videomae_data.npy"
TEST_META_PATH = "/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026/test/test_meta_data.csv"

# Constants
LABEL_MAPPING = {
    "null": 0,
    "jogging": 1,
    "jogging (rotating arms)": 2,
    "jogging (skipping)": 3,
    "jogging (sidesteps)": 4,
    "jogging (butt-kicks)": 5,
    "stretching (triceps)": 6,
    "stretching (lunging)": 7,
    "stretching (shoulders)": 8,
    "stretching (hamstrings)": 9,
    "stretching (lumbar rotation)": 10,
    "push-ups": 11,
    "push-ups (complex)": 12,
    "sit-ups": 13,
    "sit-ups (complex)": 14,
    "burpees": 15,
    "lunges": 16,
    "lunges (complex)": 17,
    "bench-dips": 18,
}

SENSORS = {
    "right_arm": ["right_arm_acc_x", "right_arm_acc_y", "right_arm_acc_z"],
    "right_leg": ["right_leg_acc_x", "right_leg_acc_y", "right_leg_acc_z"],
    "left_leg": ["left_leg_acc_x", "left_leg_acc_y", "left_leg_acc_z"],
    "left_arm": ["left_arm_acc_x", "left_arm_acc_y", "left_arm_acc_z"],
}

## Feature Engineering


In [ ]:
def extract_inertial_features(window_data):
    return (
        list(np.mean(window_data, axis=0))
        + list(np.std(window_data, axis=0))
        + list(np.min(window_data, axis=0))
        + list(np.max(window_data, axis=0))
    )


def process_single_subject(csv_path):
    filename = os.path.basename(csv_path)
    sbj_id_str = filename.replace(".csv", "")
    npy_path = os.path.join(TRAIN_VIDEO_DIR, f"{sbj_id_str}.npy")

    if not os.path.exists(npy_path):
        return pd.DataFrame()

    # low_memory=False prevents DtypeWarnings
    df = pd.read_csv(csv_path, low_memory=False)
    df["label"] = df["label"].fillna("null")
    video_feats = np.load(npy_path)
    subject_rows = []

    # 50% overlap sliding window
    for start_idx in range(0, len(df) - 50 + 1, 25):
        window_df = df.iloc[start_idx : start_idx + 50]
        label = window_df["label"].mode()[0]

        start_f = int((start_idx / 50) * 30)
        center_f = start_f + 15
        if center_f + 8 > len(video_feats):
            break

        vid_p = np.mean(video_feats[center_f - 7 : center_f + 8, :], axis=0)

        for name, cols in SENSORS.items():
            row = {"sbj_id": sbj_id_str, "sensor_location": name, "label": label}
            in_f = extract_inertial_features(window_df[cols].values)
            for i, v in enumerate(in_f):
                row[f"in_{i}"] = v
            for i, v in enumerate(vid_p):
                row[f"vi_{i}"] = v
            subject_rows.append(row)

    return pd.DataFrame(subject_rows)

## Data Processing


In [ ]:
print("Extracting Training Data...")
csv_files = glob(os.path.join(TRAIN_INERTIAL_DIR, "*.csv"))
results = Parallel(n_jobs=-1)(delayed(process_single_subject)(f) for f in csv_files)

train_df = pd.concat(results, ignore_index=True)
train_df["target"] = train_df["label"].map(LABEL_MAPPING)
train_df["sensor_location"] = train_df["sensor_location"].astype("category")

print(f"Training data shape: {train_df.shape}")

## Target Distribution


In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(
    data=train_df,
    y="label",
    order=train_df["label"].value_counts().index,
    palette="viridis",
)
plt.title("Target Distribution (Class Imbalance Check)", fontsize=14, fontweight="bold")
plt.xlabel("Count")
plt.ylabel("Activity Label")
plt.tight_layout()
plt.show()

## Correlation Heatmap


In [ ]:
plt.figure(figsize=(10, 8))
# Plot correlation for a subset of inertial features to keep it readable
subset_cols = [col for col in train_df.columns if col.startswith("in_")][:15]
corr_matrix = train_df[subset_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap (Top Inertial Features)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Feature Selection


In [ ]:
print("Identifying Top 300 Features...")
X_full = train_df.drop(columns=["sbj_id", "label", "target"])
y_full = train_df["target"]
groups = train_df["sbj_id"]

probe = CatBoostClassifier(
    iterations=200,
    depth=4,
    task_type="GPU",
    verbose=0,
    random_seed=42
)
probe.fit(X_full, y_full, cat_features=["sensor_location"])

feat_imp = pd.DataFrame({
    "feature": X_full.columns,
    "importance": probe.get_feature_importance()
    })
final_feature_list = (
    feat_imp.sort_values("importance", ascending=False)
    .head(300)["feature"]
    .tolist()
)

if "sensor_location" not in final_feature_list:
    final_feature_list.append("sensor_location")

X_selected = X_full[final_feature_list]

X_selected

## Feature Importance


In [ ]:
feature_importance = probe.get_feature_importance()
sorted_idx = np.argsort(feature_importance)[::-1][:20]

plt.figure(figsize=(10, 8))
sns.barplot(
    x=feature_importance[sorted_idx],
    y=X_full.columns[sorted_idx],
    palette="magma"
)
plt.title("CatBoost Top 20 Feature Importances", fontsize=14, fontweight="bold")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

## Ensemble Training


In [ ]:
best_params = {
    "iterations": 3000,
    "learning_rate": 0.04778,
    "depth": 4,
    "l2_leaf_reg": 3.710,
    "random_strength": 3.127,
    "bagging_temperature": 0.267,
    "task_type": "GPU",
    "loss_function": "MultiClass",
    "eval_metric": "TotalF1:average=Macro",
    "od_type": "Iter",
    "od_wait": 100,
    "verbose": 100,
    "random_seed": 42,
}

final_models = []
oof_scores = []
gkf = GroupKFold(n_splits=5)

for fold, (t_idx, v_idx) in enumerate(gkf.split(X_selected, y_full, groups)):
    print(f">>>>> Training Fold {fold+1}/5 ...")
    xt, yt = X_selected.iloc[t_idx], y_full.iloc[t_idx]
    xv, yv = X_selected.iloc[v_idx], y_full.iloc[v_idx]

    model = CatBoostClassifier(**best_params)
    model.fit(
        xt,
        yt,
        cat_features=["sensor_location"],
        early_stopping_rounds=50,
        eval_set=(xv, yv),
        use_best_model=True,
    )

    preds = model.predict(xv).flatten()
    score = f1_score(yv, preds, average="macro")
    oof_scores.append(score)
    final_models.append(model)
    print(f"   ✅ Fold {fold+1} Complete | Macro F1: {score:.4f} (Best Iter: {model.get_best_iteration()})")

print(f"\nOverall CV Macro F1: {np.mean(oof_scores):.4f}\n")

## Confusion Matrix


In [ ]:
cm = confusion_matrix(yv, preds)
class_names = [k for k, v in sorted(LABEL_MAPPING.items(), key=lambda item: item[1])]

fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Blues", ax=ax, xticks_rotation="vertical")
plt.title("Validation Confusion Matrix (Fold 5)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## Learning Curve


In [ ]:
evals_result = model.get_evals_result()
train_loss = evals_result["learn"]["MultiClass"]
valid_loss = evals_result["validation"]["MultiClass"]

plt.figure(figsize=(10, 5))
plt.plot(train_loss, label="Train Logloss", color="blue", linewidth=2)
plt.plot(valid_loss, label="Validation Logloss", color="orange", linewidth=2)
plt.title("Learning Curve (Logloss over Iterations)", fontsize=14, fontweight="bold")
plt.xlabel("Iterations")
plt.ylabel("MultiClass Logloss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## Inference & Submission


In [ ]:
# Clear memory before final inference
del train_df, X_full, y_full, xt, xv, yt, yv, X_selected, results
gc.collect()

In [ ]:
# Process Test Data
test_in = np.load(TEST_INERTIAL_PATH)
test_vid = np.load(TEST_VIDEO_PATH)
test_m = pd.read_csv(TEST_META_PATH)

# Column detection for metadata
loc_col = next(
    (
        c
        for c in ["sensor_location", "inertial_sensor_location", "location"]
        if c in test_m.columns
    ),
    None,
)
if loc_col is None:
    raise KeyError(f"Could not find location column in {test_m.columns}")

test_rows = []
for i in range(len(test_m)):
    in_data = test_in[i]
    in_f = (
        list(np.mean(in_data, axis=0))
        + list(np.std(in_data, axis=0))
        + list(np.min(in_data, axis=0))
        + list(np.max(in_data, axis=0))
    )

    vid_p = np.mean(test_vid[i], axis=1)

    # Use the detected loc_col
    row = {"sensor_location": test_m.iloc[i][loc_col]}
    for j, v in enumerate(in_f):
        row[f"in_{j}"] = v
    for j, v in enumerate(vid_p):
        row[f"vi_{j}"] = v
    test_rows.append(row)

X_test = pd.DataFrame(test_rows)[final_feature_list]
X_test["sensor_location"] = X_test["sensor_location"].astype("category")

# Average 5-Fold Predictions
ensemble_probs = np.zeros((len(X_test), 19))
for model in final_models:
    ensemble_probs += model.predict_proba(X_test)

final_preds = np.argmax(ensemble_probs, axis=1)

submission = pd.DataFrame({
    "id": test_m["id"],
    "target_feature": final_preds
})

submission.to_csv("submission.csv", index=False)
print("✅ SUCCESS! `submission.csv` is saved.")

In [ ]:
pd.read_csv("/kaggle/working/submission.csv")